In [ ]:
!pip install "setuptools<71.0.0"

In [109]:
import time

now = time.time()

In [110]:
import torch
from typing import Dict

from tdc.single_pred import Develop

import networkx as nx
import graphein.protein as gp
from graphein.ml.conversion import GraphFormatConvertor
from graphein.ml import InMemoryProteinGraphDataset, ProteinGraphDataset

In [111]:
data = Develop(name = 'SAbDab_Chen')
split = data.get_split()
# split["train"] = split["train"].iloc[30]
# split["valid"] = split["valid"].iloc[30]
# split["test"] = split["test"].iloc[30]

print(split)

Found local copy...
Loading...
Done!


{'train':      Antibody_ID                                           Antibody  Y
0           12e8  ['EVQLQQSGAEVVRSGASVKLSCTASGFNIKDYYIHWVKQRPEKG...  0
1           15c8  ['EVQLQQSGAELVKPGASVKLSCTASGFNIKDTYMHWVKQKPEQG...  0
2           1a0q  ['EVQLQESDAELVKPGASVKISCKASGYTFTDHVIHWVKQKPEQG...  1
3           1a14  ['QVQLQQSGAELVKPGASVRMSCKASGYTFTNYNMYWVKQSPGQG...  0
4           1a2y  ['QVQLQESGPGLVAPSQSLSITCTVSGFSLTGYGVNWVRQPPGKG...  0
...          ...                                                ... ..
1681        6rcs  ['QVQLVQSGAEVKKPGASVRVSCKASGYTFTSYGISWVRQAPGQG...  0
1682        6s5a  ['EVKLLESGGGLVQPGGSLKLSCAASGFDFSRYWMNWVRQAPGKG...  0
1683        6u1t  ['EVQLVESGGGLVKPGGSLKLSCAASGFTFSSYDMSWVRQTPEKR...  0
1684        7fab  ['AVQLEQSGPGLVRPSQTLSLTCTVSGTSFDDYYWTWVRQPPGRG...  0
1685        8fab  ['AVKLVQAGGGVVQPGRSLRLSCIASGFTFSNYGMHWVRQAPGKG...  0

[1686 rows x 3 columns], 'valid':     Antibody_ID                                           Antibody  Y
0          1cfq  ['QDQLQQSGAELVRP

In [112]:
from graphein.protein.utils import get_obsolete_mapping

obs = get_obsolete_mapping()

train_obs = [t for t in split["train"]["Antibody_ID"] if t in obs.keys()]
valid_obs = [t for t in split["valid"]["Antibody_ID"] if t in obs.keys()]
test_obs = [t for t in split["test"]["Antibody_ID"] if t in obs.keys()]

print(train_obs)
print(valid_obs)
print(test_obs)

['1om3', '1zls', '1zlu', '1zlw', '2f5a', '3l5y', '3qot', '3rvv', '3rvw', '3rvx', '3wxw', '4nx3', '4pp2', '4x4y', '5e99', '5kmv', '5usi', '6erx']
[]
['3wxv', '1zlv', '1op3', '3qos']


In [113]:
# If you want, you can get the PDB IDs of the new structure that replaces the obsolete entry
print("Replacement PDBs: ", [obs[t] for t  in train_obs])

# However, in this instance we will simply remove the obsolete entries from the train and test sets.
split["train"] = split["train"].loc[~split["train"]["Antibody_ID"].isin(train_obs)]
split["test"] = split["test"].loc[~split["test"]["Antibody_ID"].isin(test_obs)]

Replacement PDBs:  ['6n32', '6msy', '6mub', '6mnf', '2pr4', '4ps4', '5i18', '5vpl', '5vpg', '5vph', '6ks1', '4web', '5vco', '6dn0', '5ihu', '5vzx', '6b9j', '6fxn']


In [114]:
# Convert labels to tensors
def get_label_map(split_name: str) -> Dict[str, torch.Tensor]:
    return dict(zip(split[split_name].Antibody_ID, split[split_name].Y.apply(torch.tensor)))

train_labels = get_label_map("train")
valid_labels = get_label_map("valid")
test_labels = get_label_map("test")
print(len(train_labels))

1668


In [115]:
print(split['train'])


     Antibody_ID                                           Antibody  Y
0           12e8  ['EVQLQQSGAEVVRSGASVKLSCTASGFNIKDYYIHWVKQRPEKG...  0
1           15c8  ['EVQLQQSGAELVKPGASVKLSCTASGFNIKDTYMHWVKQKPEQG...  0
2           1a0q  ['EVQLQESDAELVKPGASVKISCKASGYTFTDHVIHWVKQKPEQG...  1
3           1a14  ['QVQLQQSGAELVKPGASVRMSCKASGYTFTNYNMYWVKQSPGQG...  0
4           1a2y  ['QVQLQESGPGLVAPSQSLSITCTVSGFSLTGYGVNWVRQPPGKG...  0
...          ...                                                ... ..
1681        6rcs  ['QVQLVQSGAEVKKPGASVRVSCKASGYTFTSYGISWVRQAPGQG...  0
1682        6s5a  ['EVKLLESGGGLVQPGGSLKLSCAASGFDFSRYWMNWVRQAPGKG...  0
1683        6u1t  ['EVQLVESGGGLVKPGGSLKLSCAASGFTFSSYDMSWVRQTPEKR...  0
1684        7fab  ['AVQLEQSGPGLVRPSQTLSLTCTVSGTSFDDYYWTWVRQPPGRG...  0
1685        8fab  ['AVKLVQAGGGVVQPGRSLRLSCIASGFTFSNYGMHWVRQAPGKG...  0

[1668 rows x 3 columns]


In [116]:
def fit(g: nx.Graph) -> nx.Graph:
    g_config = g.graph['config']
    g_pdb_code = g.graph['pdb_code']

    g.graph.clear()
    g.graph['config'] = g_config
    g.graph['pdb_code'] = g_pdb_code

    return g

In [117]:
from functools import partial

config = gp.ProteinGraphConfig(
        node_metadata_functions=[gp.amino_acid_one_hot],
        edge_construction_functions=[partial(gp.add_distance_threshold, threshold=6, long_interaction_threshold=0)],
        graph_metadata_functions=[fit]
)


In [118]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg", columns=["coords", "edge_index", "amino_acid_one_hot"])

In [119]:
from tqdm import tqdm
from pkg.PDBGraphStore import PDBGraphStore as store
from pkg.MemoryMeasuring import MemoryMeasuring as m

import traceback
import os

In [120]:
pdb_store = store()
dataset_list = list(data.get_data()['Antibody_ID'])

In [121]:
pdb_data_path = '../../data/pdb_files'

pdb_codes = ()

for pdb_code in tqdm(dataset_list):
    if os.path.exists(f'{pdb_data_path}/{pdb_code}.pdb'):
        pdb_file = os.path.abspath(f'{pdb_data_path}/{pdb_code}.pdb')
    else:
        try:
            pdb_file = gp.download_pdb(pdb_code, f'{pdb_data_path}/')
        except Exception:
            print(f"Erro em {pdb_code}")
            traceback.print_exc()  
    try:
        g = gp.construct_graph(path=pdb_file, pdb_code=pdb_code, config=config)

        pdb_store.insert({pdb_code: g})
    except Exception:
        print(f"Erro em {pdb_code}")
        traceback.print_exc()  

g = None
print(pdb_store)

  0%|          | 0/2409 [00:00<?, ?it/s]

Output()

  0%|          | 1/2409 [00:00<15:13,  2.64it/s]

Output()

  0%|          | 2/2409 [00:00<09:52,  4.06it/s]

...

Output()

100%|█████████▉| 2408/2409 [12:12<00:00,  4.95it/s]

Output()

100%|██████████| 2409/2409 [12:12<00:00,  3.29it/s]

PDBGraphStore with 2408 pdbs


In [122]:
memo = m(pdb_store)

In [123]:
print(memo.total_memory())

3403.949592590332


In [124]:
train_ds = []

for pdb_code in tqdm(split['train']['Antibody_ID']):
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}
        aux[pdb_code] = convertor(g)
        aux[pdb_code].graph_y = train_labels[pdb_code]
        train_ds.append(aux[pdb_code])

    except Exception:
        print(f"Erro em {pdb_code}")
        traceback.print_exc()  

 47%|████▋     | 791/1668 [00:57<01:33,  9.41it/s]Traceback (most recent call last):
  File "/tmp/ipykernel_38223/2909225962.py", line 5, in <module>
    g = pdb_store.extract(pdb_code)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 369, in extract
    return extract()
           ^^^^^^^^^
  File "/home/jovyan/app/src/PDBGraphStore_notebooks/../pkg/PDBGraphStore.py", line 352, in extract
    pdb_id = self.__body_parts["pdb_code_to_id"][pdb_upper]
             ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '4GXV'
 48%|████▊     | 796/1668 [00:57<01:03, 13.74it/s]

Erro em 4gxv


100%|██████████| 1668/1668 [02:05<00:00, 13.30it/s]


In [125]:
valid_ds = []

for pdb_code in tqdm(split['valid']['Antibody_ID']):
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}
        aux[pdb_code] = convertor(g)
        aux[pdb_code].graph_y = valid_labels[pdb_code]
        valid_ds.append(aux[pdb_code])

    except Exception:
        print(f"Erro em {pdb_code}")
        traceback.print_exc()  

100%|██████████| 241/241 [00:19<00:00, 12.13it/s]


In [126]:
test_ds = []

for pdb_code in tqdm(split['test']['Antibody_ID']):
    try:
        g = pdb_store.extract(pdb_code)
        aux = {}
        aux[pdb_code] = convertor(g)
        aux[pdb_code].graph_y = test_labels[pdb_code]
        test_ds.append(aux[pdb_code])

    except Exception:
        print(f"Erro em {pdb_code}")
        traceback.print_exc()  

100%|██████████| 478/478 [00:37<00:00, 12.63it/s]


In [127]:
from torch_geometric.loader import DataLoader

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=False, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=16, drop_last=True)

In [128]:
for b in valid_loader:
    print(b)
    break

DataBatch(edge_index=[2, 70394], node_id=[16], amino_acid_one_hot=[11748, 20], coords=[11748, 3], num_nodes=11748, graph_y=[16], batch=[11748], ptr=[17])


In [129]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import itertools

In [130]:
"""EGNN Implementation from Satorras et al. https://github.com/vgsatorras/egnn"""

class E_GCL(nn.Module):
    """
    E(n) Equivariant Convolutional Layer
    re
    """

    def __init__(self, input_nf, output_nf, hidden_nf, edges_in_d=0, act_fn=nn.SiLU(), residual=True, attention=False, normalize=False, coords_agg='mean', tanh=False):
        super(E_GCL, self).__init__()
        input_edge = input_nf * 2
        self.residual = residual
        self.attention = attention
        self.normalize = normalize
        self.coords_agg = coords_agg
        self.tanh = tanh
        self.epsilon = 1e-8
        edge_coords_nf = 1

        self.edge_mlp = nn.Sequential(
            nn.Linear(input_edge + edge_coords_nf + edges_in_d, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, hidden_nf),
            act_fn)

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf))

        layer = nn.Linear(hidden_nf, 1, bias=False)
        torch.nn.init.xavier_uniform_(layer.weight, gain=0.001)

        coord_mlp = [nn.Linear(hidden_nf, hidden_nf)]
        coord_mlp.append(act_fn)
        coord_mlp.append(layer)
        if self.tanh:
            coord_mlp.append(nn.Tanh())
        self.coord_mlp = nn.Sequential(*coord_mlp)

        if self.attention:
            self.att_mlp = nn.Sequential(
                nn.Linear(hidden_nf, 1),
                nn.Sigmoid())

    def edge_model(self, source, target, radial, edge_attr):
        if edge_attr is None:  # Unused.
            out = torch.cat([source, target, radial], dim=1)
        else:
            out = torch.cat([source, target, radial, edge_attr], dim=1)
        out = self.edge_mlp(out)
        if self.attention:
            att_val = self.att_mlp(out)
            out = out * att_val
        return out

    def node_model(self, x, edge_index, edge_attr, node_attr):
        row, col = edge_index
        agg = unsorted_segment_sum(edge_attr, row, num_segments=x.size(0))
        if node_attr is not None:
            agg = torch.cat([x, agg, node_attr], dim=1)
        else:
            agg = torch.cat([x, agg], dim=1)
        out = self.node_mlp(agg)
        if self.residual:
            out = x + out
        return out, agg

    def coord_model(self, coord, edge_index, coord_diff, edge_feat):
        row, col = edge_index
        trans = coord_diff * self.coord_mlp(edge_feat)
        if self.coords_agg == 'sum':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0))
        elif self.coords_agg == 'mean':
            agg = unsorted_segment_mean(trans, row, num_segments=coord.size(0))
        else:
            raise Exception('Wrong coords_agg parameter' % self.coords_agg)
        coord += agg
        return coord

    def coord2radial(self, edge_index, coord):
        row, col = edge_index
        coord_diff = coord[row] - coord[col]
        radial = torch.sum(coord_diff**2, 1).unsqueeze(1)

        if self.normalize:
            norm = torch.sqrt(radial).detach() + self.epsilon
            coord_diff = coord_diff / norm

        return radial, coord_diff

    def forward(self, h, edge_index, coord, edge_attr=None, node_attr=None):
        row, col = edge_index
        radial, coord_diff = self.coord2radial(edge_index, coord)

        edge_feat = self.edge_model(h[row], h[col], radial, edge_attr)
        coord = self.coord_model(coord, edge_index, coord_diff, edge_feat)
        h, agg = self.node_model(h, edge_index, edge_feat, node_attr)

        return h, coord, edge_attr

class EGNN(nn.Module):
    def __init__(self, in_node_nf, hidden_nf, out_node_nf, in_edge_nf=0, device='cpu', act_fn=nn.SiLU(), n_layers=4, residual=True, attention=False, normalize=False, tanh=False):
        '''
        :param in_node_nf: Number of features for 'h' at the input
        :param hidden_nf: Number of hidden features
        :param out_node_nf: Number of features for 'h' at the output
        :param in_edge_nf: Number of features for the edge features
        :param device: Device (e.g. 'cpu', 'cuda:0',...)
        :param act_fn: Non-linearity
        :param n_layers: Number of layer for the EGNN
        :param residual: Use residual connections, we recommend not changing this one
        :param attention: Whether using attention or not
        :param normalize: Normalizes the coordinates messages such that:
                    instead of: x^{l+1}_i = x^{l}_i + Σ(x_i - x_j)phi_x(m_ij)
                    we get:     x^{l+1}_i = x^{l}_i + Σ(x_i - x_j)phi_x(m_ij)/||x_i - x_j||
                    We noticed it may help in the stability or generalization in some future works.
                    We didn't use it in our paper.
        :param tanh: Sets a tanh activation function at the output of phi_x(m_ij). I.e. it bounds the output of
                        phi_x(m_ij) which definitely improves in stability but it may decrease in accuracy.
                        We didn't use it in our paper.
        '''

        super(EGNN, self).__init__()
        self.hidden_nf = hidden_nf
        self.device = device
        self.n_layers = n_layers
        self.embedding_in = nn.Linear(in_node_nf, self.hidden_nf)
        self.embedding_out = nn.Linear(self.hidden_nf, out_node_nf)
        for i in range(n_layers):
            self.add_module("gcl_%d" % i, E_GCL(self.hidden_nf, self.hidden_nf, self.hidden_nf, edges_in_d=in_edge_nf,
                                                act_fn=act_fn, residual=residual, attention=attention,
                                                normalize=normalize, tanh=tanh))
        self.to(self.device)

    def forward(self, h, x, edges, edge_attr):
        h = self.embedding_in(h)
        for i in range(self.n_layers):
            h, x, _ = self._modules["gcl_%d" % i](h, edges, x, edge_attr=edge_attr)
        h = self.embedding_out(h)
        return h, x


def unsorted_segment_sum(data, segment_ids, num_segments):
    result_shape = (num_segments, data.size(1))
    result = data.new_full(result_shape, 0)  # Init empty result tensor.
    segment_ids = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result.scatter_add_(0, segment_ids, data)
    return result


def unsorted_segment_mean(data, segment_ids, num_segments):
    result_shape = (num_segments, data.size(1))
    segment_ids = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result = data.new_full(result_shape, 0)  # Init empty result tensor.
    count = data.new_full(result_shape, 0)
    result.scatter_add_(0, segment_ids, data)
    count.scatter_add_(0, segment_ids, torch.ones_like(data))
    return result / count.clamp(min=1)


def get_edges(n_nodes):
    rows, cols = [], []
    for i, j in itertools.product(range(n_nodes), range(n_nodes)):
        if i != j:
            rows.append(i)
            cols.append(j)

    return [rows, cols]


def get_edges_batch(n_nodes, batch_size):
    edges = get_edges(n_nodes)
    edge_attr = torch.ones(len(edges[0]) * batch_size, 1)
    edges = [torch.LongTensor(edges[0]), torch.LongTensor(edges[1])]
    
    if batch_size == 1:
        return edges, edge_attr
    elif batch_size > 1:
        rows, cols = [], []
        for i in range(batch_size):
            rows.append(edges[0] + n_nodes * i)
            cols.append(edges[1] + n_nodes * i)
        edges = [torch.cat(rows), torch.cat(cols)]
    return edges, edge_attr

In [131]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch.nn.functional import binary_cross_entropy_with_logits, mse_loss
from torch_geometric.nn import global_add_pool
import pytorch_lightning as pl
#from torchmetrics import F1Score, Accuracy, AUROC
from pytorch_lightning.loggers import WandbLogger


class SimpleEGNN(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = EGNN(
            in_node_nf=20,
            out_node_nf=32,
            in_edge_nf=0,
            hidden_nf=32,
            n_layers=2,
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
        self.loss = binary_cross_entropy_with_logits

    def configure_loss(self, name: str):
        """Return the loss function based on the config."""
        return self.loss

    # --- Forward pass
    def forward(self, x):
        x.aa = torch.stack(
            [torch.tensor(a) for a in x.amino_acid_one_hot]
        ).float().cuda()

        x.c = torch.stack(
            [torch.tensor(a).squeeze(0) for a in x.coords]
        ).float().cuda()
        
        feats, coords = self.model(
            h=x.aa,
            x=x.c,
            edges=x.edge_index,
            edge_attr=None,
        )
        feats = global_add_pool(feats, x.batch)
        return self.decoder(feats)

    def training_step(self, batch: Data, batch_idx) -> torch.Tensor:
        x = batch
        y = batch.graph_y.unsqueeze(1).float()
        y_hat = self(x)

        loss = self.loss(y_hat, y)
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch
        y = batch.graph_y.unsqueeze(1).float()
        y_hat = self(batch)

    def test_step(self, batch, batch_idx):
        x = batch
        y = batch.graph_y.unsqueeze(1).float()
        y_hat = self(x)
        loss = self.loss(y_hat, y)

        y_pred_softmax = torch.log_softmax(y_hat, dim = 1)
        y_pred_tags = torch.argmax(y_pred_softmax, dim = 1)
        self.log("test_loss", loss)
        #return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.Adam(params=self.parameters(), lr=0.001)

In [132]:
trainer = pl.Trainer(
    accelerator='auto',
    benchmark=True,
    deterministic=False,
    num_sanity_val_steps=0,
    max_epochs=10,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [133]:
model = SimpleEGNN()
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ EGNN       │ 16.5 K │ train │     0 │
│ 1 │ decoder │ Sequential │  1.1 K │ train │     0 │
└───┴─────────┴────────────┴────────┴───────┴───────┘

Trainable params: 17.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 17.6 K                                                                                               
Total estimated model params size (MB): 0.070                                                                      
Modules in train mode: 28                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


In [134]:
trainer.test(model, test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.7803594470024109     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.7803594470024109}]

In [135]:
with open('tdc_time.txt', 'a') as f:
    f.write(f'\n{str(time.time() - now)}')